In [1]:
import fitz
import os

PDF_DIR = "../data/pdfs"
TEXT_DIR = "../data/extracted"

os.makedirs(TEXT_DIR, exist_ok=True)

def extract_text_from_pdfs():
    for pdf in os.listdir(PDF_DIR):
        if not pdf.endswith(".pdf"):
            continue

        doc = fitz.open(os.path.join(PDF_DIR, pdf))
        text = ""

        for page in doc:
            text += page.get_text()

        out_path = os.path.join(TEXT_DIR, pdf.replace(".pdf", ".txt"))
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)

        print(f"Extracted text → {out_path}")

extract_text_from_pdfs()


Extracted text → ../data/extracted\paper_1.txt
Extracted text → ../data/extracted\paper_2.txt
Extracted text → ../data/extracted\paper_3.txt


In [2]:
sample_txt = os.path.join(TEXT_DIR, os.listdir(TEXT_DIR)[0])

with open(sample_txt, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:1000])

Proceedings of the 57th Annual Meeting of the Association for Computational Linguistics, pages 3645–3650
Florence, Italy, July 28 - August 2, 2019. c⃝2019 Association for Computational Linguistics
3645
Energy and Policy Considerations for Deep Learning in NLP
Emma Strubell
Ananya Ganesh
Andrew McCallum
College of Information and Computer Sciences
University of Massachusetts Amherst
{strubell, aganesh, mccallum}@cs.umass.edu
Abstract
Recent progress in hardware and methodol-
ogy for training neural networks has ushered
in a new generation of large networks trained
on abundant data.
These models have ob-
tained notable gains in accuracy across many
NLP tasks. However, these accuracy improve-
ments depend on the availability of exception-
ally large computational resources that neces-
sitate similarly substantial energy consump-
tion. As a result these models are costly to
train and develop, both ﬁnancially, due to the
cost of hardware and electricity or cloud com-
pute time, and environm

In [3]:
import re

def clean_text(text: str) -> str:
    # Fix broken hyphenated words
    text = re.sub(r"-\n", "", text)

    # Reduce excessive newlines
    text = re.sub(r"\n+", "\n", text)

    # Merge broken sentences
    lines = text.split("\n")
    merged = []

    for line in lines:
        if line.strip().isupper():
            merged.append("\n" + line.strip() + "\n")
        else:
            merged.append(line.strip() + " ")

    text = "".join(merged)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [4]:
cleaned_text = clean_text(raw_text)
print(cleaned_text[:1000])

Proceedings of the 57th Annual Meeting of the Association for Computational Linguistics, pages 3645–3650 Florence, Italy, July 28 - August 2, 2019. c⃝2019 Association for Computational Linguistics 3645 Energy and Policy Considerations for Deep Learning in NLP Emma Strubell Ananya Ganesh Andrew McCallum College of Information and Computer Sciences University of Massachusetts Amherst {strubell, aganesh, mccallum}@cs.umass.edu Abstract Recent progress in hardware and methodology for training neural networks has ushered in a new generation of large networks trained on abundant data. These models have obtained notable gains in accuracy across many NLP tasks. However, these accuracy improvements depend on the availability of exceptionally large computational resources that necessitate similarly substantial energy consumption. As a result these models are costly to train and develop, both ﬁnancially, due to the cost of hardware and electricity or cloud compute time, and environmentally, due t

In [5]:
SECTION_PATTERNS = {
    "abstract": r"\babstract\b",
    "introduction": r"\bintroduction\b",
    "methodology": r"\b(methodology|methods|approach|model|architecture)\b",
    "experiments": r"\b(experiments|experimental setup|evaluation)\b",
    "results": r"\b(results|analysis)\b",
    "conclusion": r"\b(conclusion|discussion|future work)\b"
}

In [6]:
def split_sections(text):
    sections = {key: "" for key in SECTION_PATTERNS}
    sections["full_text"] = text

    text_lower = text.lower()
    indices = {}

    for section, pattern in SECTION_PATTERNS.items():
        match = re.search(pattern, text_lower)
        if match:
            indices[section] = match.start()

    if not indices:
        return sections

    sorted_sections = sorted(indices.items(), key=lambda x: x[1])

    for i, (section, start) in enumerate(sorted_sections):
        end = sorted_sections[i + 1][1] if i + 1 < len(sorted_sections) else len(text)
        content = text[start:end].strip()

        # Ignore tiny accidental matches
        if len(content.split()) > 100:
            sections[section] = content

    return sections

In [7]:
def validate_sections(sections):
    report = {
        "abstract_ok": False,
        "methodology_ok": False,
        "results_ok": False,
        "overall_valid": False,
        "issues": []
    }

    # Abstract (relaxed for NLP papers)
    if sections.get("abstract") and len(sections["abstract"].split()) >= 80:
        report["abstract_ok"] = True
    else:
        report["issues"].append("Abstract missing or too short")

    # Methodology
    if sections.get("methodology") and len(sections["methodology"].split()) >= 300:
        report["methodology_ok"] = True
    else:
        report["issues"].append("Methodology missing or too short")

    # Results
    if sections.get("results") and len(sections["results"].split()) >= 300:
        report["results_ok"] = True
    else:
        report["issues"].append("Results missing or too short")

    # Overall logic (abstract OR strong content)
    if report["abstract_ok"] or (report["methodology_ok"] and report["results_ok"]):
        report["overall_valid"] = True

    return report


In [8]:
def recover_abstract(text):
    if "abstract" in text.lower():
        idx = text.lower().find("abstract")
        return text[idx: idx + 1500]  # first ~250 words
    return ""

In [9]:
def normalize_sections(sections):
    # If methodology missing, fallback to introduction
    if not sections.get("methodology"):
        sections["methodology"] = sections.get("introduction", "")

    # If results missing, fallback to experiments
    if not sections.get("results"):
        sections["results"] = sections.get("experiments", "")

    # Abstract fallback
    if not sections.get("abstract"):
        sections["abstract"] = recover_abstract(sections["full_text"])

    return sections

In [10]:
sections = split_sections(cleaned_text)
sections = normalize_sections(sections)
validation = validate_sections(sections)

In [11]:
validation

{'abstract_ok': True,
 'methodology_ok': False,
 'results_ok': True,
 'overall_valid': True,
 'issues': ['Methodology missing or too short']}

In [12]:
def extract_key_findings(sections):
    findings = []

    text = sections.get("results", "") + sections.get("conclusion", "")
    sentences = re.split(r'(?<=[.!?])\s+', text)

    KEY_PHRASES = [
        "we find",
        "our results",
        "results show",
        "results indicate",
        "demonstrate",
        "outperform",
        "significant",
        "improves",
        "achieves"
    ]

    for s in sentences:
        s_low = s.lower()
        if any(k in s_low for k in KEY_PHRASES) and len(s.split()) > 12:
            findings.append(s.strip())

    return findings[:10]

In [13]:
def build_paper_profile(paper_id, sections):
    return {
        "paper_id": paper_id,
        "abstract": sections.get("abstract", "")[:600],
        "methodology": sections.get("methodology", "")[:800],
        "key_findings": extract_key_findings(sections)
    }

In [14]:
paper_profiles = []

for idx, txt in enumerate(os.listdir(TEXT_DIR)):
    with open(os.path.join(TEXT_DIR, txt), "r", encoding="utf-8") as f:
        raw = f.read()

    cleaned = clean_text(raw)
    secs = normalize_sections(split_sections(cleaned))
    profile = build_paper_profile(f"paper_{idx+1}", secs)
    paper_profiles.append(profile)

len(paper_profiles)

3

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def compare_papers_semantic(paper_profiles, threshold=0.35):
    findings = []
    meta = []

    for paper in paper_profiles:
        for f in paper["key_findings"]:
            findings.append(f)
            meta.append(paper["paper_id"])

    if len(findings) < 2:
        return {"common_findings": {}, "paper_wise_findings": {}}

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform(findings)

    similarity = cosine_similarity(tfidf)

    common = {}

    for i in range(len(findings)):
        for j in range(i + 1, len(findings)):
            if similarity[i][j] >= threshold and meta[i] != meta[j]:
                key = findings[i][:80]
                common.setdefault(key, set()).update([meta[i], meta[j]])

    return {
        "common_findings": {k: list(v) for k, v in common.items()},
        "paper_wise_findings": {
            p["paper_id"]: p["key_findings"] for p in paper_profiles
        }
    }

In [19]:
THEMES = {
    "performance_improvement": [
        "improves", "outperforms", "accuracy", "bleu", "state-of-the-art", "achieves"
    ],
    "bias_fairness": [
        "bias", "fairness", "political", "ideological", "partisan"
    ],
    "dataset_artifacts": [
        "dataset", "artifacts", "benchmark", "spurious", "corpus"
    ],
    "generalization_limits": [
        "overestimated", "fails", "limitations", "does not generalize"
    ]
}

def theme_based_comparison(paper_profiles):
    theme_map = {theme: [] for theme in THEMES}

    for paper in paper_profiles:
        pid = paper["paper_id"]
        for finding in paper["key_findings"]:
            f_low = finding.lower()
            for theme, keywords in THEMES.items():
                if any(k in f_low for k in keywords):
                    theme_map[theme].append(pid)
                    break

    # keep only cross-paper themes
    theme_map = {
        t: list(set(pids))
        for t, pids in theme_map.items()
        if len(set(pids)) > 1
    }

    return theme_map

In [20]:
theme_comparison = theme_based_comparison(paper_profiles)
theme_comparison

{'performance_improvement': ['paper_1', 'paper_3'],
 'bias_fairness': ['paper_2', 'paper_3'],
 'dataset_artifacts': ['paper_2', 'paper_3']}